# MuJoCo URDF Import
Onshape can export assemblies in the [Universal Robotic Description Format (URDF)](https://wiki.ros.org/urdf). MuJoCo only has limited support for working with these files directly [(docs section)](https://mujoco.readthedocs.io/en/stable/modeling.html#curdf), so instead parse some key information from it to create a MuJoCo-native MJCF XML file.

## Config

In [1]:
from pathlib import Path

import mujoco as mj

# ONSHAPE_ASSEMBLY = "assembly_simplified"
# OUTPUT = "robot.xml"
ONSHAPE_ASSEMBLY = "assembly_bala2ish"
OUTPUT = "bala2ish.xml"

mass_override = {
	"electronics": 0.032,
	# "electronics": 0.0001
	# "wheel": 0.017,
	# "wheel_1": 0.017,
	"sensor_imu": 0,
}

# Onshape-exported geometry of wheels is not ideal, use cylinders instead

wheelsize = [45 / 1000 / 2, 14.15 / 1000 / 2]
geom_override = {
	# "wheel": dict(
	# 	type=mj.mjtGeom.mjGEOM_CYLINDER,
	# 	mass=0.025,
	# 	size=wheelsize,
	# 	zaxis=[1, 0, 0],
	# 	pos=[wheelsize[0]/4, 0, 0]
	# ),
	# "wheel_1": dict(
	# 	type=mj.mjtGeom.mjGEOM_CYLINDER,
	# 	mass=0.025,
	# 	size=[45/1000/2, 14.15/1000/2],
	# 	zaxis=[1, 0, 0],
	# 	pos=[-wheelsize[0]/4, 0, 0]
	# ),
}


if ONSHAPE_ASSEMBLY == "assembly_bala2ish":
	mass_override["wheel"] = 0.017
	mass_override["wheel_1"] = 0.017
	geom_override["chassis"] = dict(
		mass=0.127, 
		# ipos=[0, 0, 0.017]
	)


cwd = Path(".")
path_assembly = cwd / ONSHAPE_ASSEMBLY
path_urdf = path_assembly / "urdf" / (ONSHAPE_ASSEMBLY + ".urdf")
path_meshes = path_assembly / "meshes"

In [2]:
def mapl(items, func):
	return list(map(func, items))


def parse_vec(string):
	return mapl(string.split(" "), lambda x: float(x))

## Parsing

In [3]:
import xml.etree.ElementTree as ET


xml = ET.parse(path_urdf)

robot = xml.getroot()

links = {}

print("Links (bodies):")
for link in robot.findall("./link"):
	name = link.get("name")
	print(f"  {name}")
	links[name] = {"xml": link, "children": []}

print()
print("Joints:")
for joint in robot.findall("./joint"):
	joint_name = joint.get("name")
	joint_type = joint.get("type")
	if not (joint_type == "fixed" or joint_type == "continuous"):
		print(f"  '{joint_name}' unsupported mate type: {joint_type}")
		continue
	parent = joint.find("./parent").get("link")
	child = joint.find("./child").get("link")
	print(f"  '{joint_name}' {joint_type} {child} to {parent}")
	p = links.get(parent)
	mate = {
		"joint_name": joint_name,
		"joint_type": joint_type,
		"joint_axis": parse_vec(joint.find("./axis").get("xyz")),
		"origin": parse_vec(joint.find("./origin").get("xyz")),
		"rpy": parse_vec(joint.find("./origin").get("rpy")),
		"child": child,
	}
	p["children"].append(mate)


Links (bodies):
  chassis
  mark_wheel
  mark_wheel_1
  root
  sensor_imu
  wheel
  wheel_1

Joints:
  'fastened_1' fixed chassis to root
  'fastened_2' fixed sensor_imu to chassis
  'fastened_3' fixed mark_wheel_1 to wheel_1
  'fastened_4' fixed mark_wheel to wheel
  'wheel_left' continuous wheel_1 to chassis
  'wheel_right' continuous wheel to chassis


In [4]:
import math
from scipy.spatial.transform import Rotation

spec = mj.MjSpec.from_file("template.xml")

spec.compiler.meshdir = str(path_meshes)


def convert(mate, parent=spec.worldbody):
	name = mate.get("child")
	link = links.get(name)
	print(name, mate, link)

	has_rot = False
	for x in mate["rpy"]:
		if abs(x) > 1e-3:
			has_rot = True
			break

	euler=[0, 0, 0]
	if has_rot:
		euler = [ round(x/math.pi)*math.pi for x in mate["rpy"]]
		print("  joint rotation", euler)
		# print("  joint rotation", rot)
		# mate["origin"] = rot.apply(mate["origin"])
		# euler = rot
	rot = Rotation.from_euler("xyz", euler)

	inertial = link["xml"].find("./inertial")
	body: mj.MjsBody = parent.add_body(
		name=name,
		pos=rot.apply(mate["origin"]) if has_rot else mate["origin"],
		# euler=euler
		# explicitinertial=parent is not spec.worldbody,
	)
	mesh = link["xml"].find("./visual/geometry/mesh")
	color = link["xml"].find("./visual/material/color")
	if color is not None:
		print("  ", "color", color)
		color = parse_vec(color.get("rgba"))

	# if parent is spec.worldbody:
	# 	# root body
	# 	body.add_site(
	# 		name="site_root",
	# 		pos=[0, 0, geom_override["wheel"]["size"][0]],
	# 		size=[0.001, 0.001, 0.001],
	# 	)

	if mesh is not None:
		if inertial is None:
			print(f"error: mesh {name} has no inertial info")
			return

		mass = float(inertial.find("mass").get("value"))

		# origin = inertial.find("origin")
		# in_pos = parse_vec(origin.get("xyz"))
		# in_orient = parse_vec(origin.get("rpy"))

		# inertia = inertial.find("inertia")
		# inermat = []
		# for dim in ["ixx", "iyy", "izz", "ixy", "ixz", "iyz"]:
		# 	inermat.append(float(inertia.get(dim)))

		# body.mass=mass_override.get(name, mass)
		# body.position=in_pos,
		# body.orientation=in_orient,
		# body.fullinertia=inermat

		mesh_file = Path(mesh.get("filename").replace("package://", "")).relative_to(
			path_meshes
		)
		mesh_name = name
		spec.add_mesh(
			name=mesh_name, file=str(mesh_file), scale=parse_vec(mesh.get("scale"))
		)
		spec.assets[mesh_name] = str(mesh_file)
		geom_name = f"geom_{name}"
		pos = parse_vec(link["xml"].find("./visual/origin").get("xyz"))
		opts = dict(
				name=geom_name,
				rgba=color,
				type=mj.mjtGeom.mjGEOM_MESH,
				meshname=mesh_name,
				mass=mass_override.get(name, mass),
				pos=pos,
				euler=euler
			)	| geom_override.get(name, dict())

		if name.startswith("sensor_") or name.startswith("mark_"):
			opts["mass"] = 0
		# ipos = None
		# if "ipos" in opts:
		# 	ipos = opts.pop("ipos")
		body.add_geom(
			**opts
		)
		# if ipos is not None:
		# 	# body.explicitinertial = True
		# 	body.ipos = ipos

	else:
		print(f"  {name} has no geometry")


	joint_name = mate["joint_name"]
	match mate["joint_type"]:
		case "freejoint":
			body.add_freejoint(name=joint_name)
		case "fixed":
			pass  # mujoco default
		case "continuous":
			body.add_joint(
				name=f"joint_{joint_name}",
				type=mj.mjtJoint.mjJNT_HINGE,
				axis=mate["joint_axis"],
				damping=0.001,
				frictionloss=0.001,
				# damping=0,
				# frictionloss=0,
				#  armature: motor's rotational inertia reflected through the gearbox
				#            Estimated rotational inertia of N20 motor: 5e-8 Kg-m^2.
				#            With a 1:N gearbox, we multiply the inertia by N^2 to get
				#            the reflected inertia. So, 5e-8 * 30^2 = 4.5e-5 Kg-m^2.
				#            This is ~10x the wheel's inertia, so this dominates. -->
				#
				armature=4.5e-5,
				# armature=0,
			)

			#  Control torque (for a hinge joint) is determined by gear * ctrl (units
			#  of N-m). For reinforcement learning, it helps to have a normalized
			#  ctrl, so we set the limits to [-1, 1]. With the gear set to 0.05, that
			#  gives us actual limits of -0.05 to 0.05 N-m, which is a decent estimate
			#  of our N20 motor with a 1:30 gearbox.

			act = spec.add_actuator(name=f"act_{joint_name}")
			# act.damping = [0, 0, 0] # messes with dcmotor..
			# act.set_to_dcmotor(
			# 	# measured going ~1m/0.73s, 43.6mm dia wheels, = 600rpm (motor spec 500rpm at 6V, but running at 8V)
			# 	# at 30:1 gearing so shaft = 18000 rpm
			# 	# so motor constant Kv = 0.004244
			# 	# motorconst=[0.004244, 0.004244],
			# 	# motorconst=[0.00382, 0.00382], # 6V, 500rpm
			# 	# motorconst=[0.004, 0.004],
			# 	# motorconst=[0.16, 0.16],
			# 	motorconst=[0, 0],
			# 	# resistance=11.5,  # ohm
			# 	resistance=0,
			# 	nominal=[
			# 		8, 0.15*8/11.5, 8.023*2*math.pi
			# 	],
			# 	saturation=[
			# 		0,
			# 		0.15,
			# 		0,
			# 	],
			# 	inductance=[
			# 		0.0026,  # 2.6mH based on https://www.klikrobotics.com/PWMTutorial2.html
			# 		0,
			# 	],
			# 	thermal=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
			# 	cogging=[0.0, 0.0, 0.0]
			# )
			# # act.gear[0] = 30 # 30:1 gearbox
			# act.ctrlrange = [-8, 8]

			act.set_to_motor()
			act.ctrlrange = [-1, 1]
			# # act.gear[0] = 0.05 # original
			# # act.gear[0] = 0.0058 # 1:1
			# # act.gear[0] = 0.007 # fast gearing
			# # act.gear[0] = 0.031 # fastest gearing
			# # new 500rpm motors
			act.gear[0] = 0.05

			act.target = f"joint_{joint_name}"
			act.trntype = mj.mjtTrn.mjTRN_JOINT

			spec.add_sensor(
				name=f"jointvel_{joint_name}",
				type=mj.mjtSensor.mjSENS_JOINTVEL,
				objtype=mj.mjtObj.mjOBJ_JOINT,
				objname=f"joint_{joint_name}",
			)

	if name.startswith("sensor_"):
		body.add_site(name=f"site_{name}", pos=[0, 0, 0], size=[0.001, 0.001, 0.001])
		sensor_type = name.replace("sensor_", "")
		if sensor_type == "imu":
			# spec.add_sensor(name="imu_accel", type=mj.mjtSensor.mjSENS_ACCELEROMETER, site="site_sensor_imu")
			# spec.add_sensor(name="imu_gyro", type=mj.mjtSensor.mjSENS_GYRO, site="site_sensor_imu")
			# spec.add_sensor(name="imu_orientation", objtype=mj.MjsSite, objname="site_sensor_imu")

			# BNO055 provides orientation quaternion
			spec.add_sensor(
				name="imu_quat",
				type=mj.mjtSensor.mjSENS_FRAMEQUAT,
				objtype=mj.mjtObj.mjOBJ_SITE,
				objname="site_sensor_imu",
				# reftype=mj.mjtObj.mjOBJ_SITE,
				# refname="site_sensor_imu"
			)
			spec.add_sensor(
				name="imu_accel",
				type=mj.mjtSensor.mjSENS_ACCELEROMETER,
				objtype=mj.mjtObj.mjOBJ_SITE,
				objname="site_sensor_imu",
			)
			spec.add_sensor(
				name="imu_gyro",
				type=mj.mjtSensor.mjSENS_GYRO,
				objtype=mj.mjtObj.mjOBJ_SITE,
				objname="site_sensor_imu",
			)

	for c in link["children"]:
		# convert(c, body if parent is spec.worldbody else parent)
		convert(c, body)


convert(
	{
		"joint_name": "_root",
		"joint_type": "freejoint",
		# "origin": [0,0,0.5],
		"origin": [0, 0, 0],
		"rpy": [0, 0, 0],
		"child": "root",
	}
)

# privileged velocity info
spec.add_sensor(
	name="priv_vel",
	type=mj.mjtSensor.mjSENS_VELOCIMETER,
	objtype=mj.mjtObj.mjOBJ_SITE,
	objname="site_sensor_imu",
)

print()
print(spec.to_xml())

spec.encode(OUTPUT)


root {'joint_name': '_root', 'joint_type': 'freejoint', 'origin': [0, 0, 0], 'rpy': [0, 0, 0], 'child': 'root'} {'xml': <Element 'link' at 0x7c1fb498b830>, 'children': [{'joint_name': 'fastened_1', 'joint_type': 'fixed', 'joint_axis': [-0.0, -0.0, -1.0], 'origin': [0.0, 0.0, 0.0], 'rpy': [0.0, -0.0, 0.0], 'child': 'chassis'}]}
  root has no geometry
chassis {'joint_name': 'fastened_1', 'joint_type': 'fixed', 'joint_axis': [-0.0, -0.0, -1.0], 'origin': [0.0, 0.0, 0.0], 'rpy': [0.0, -0.0, 0.0], 'child': 'chassis'} {'xml': <Element 'link' at 0x7c1fb498abb0>, 'children': [{'joint_name': 'fastened_2', 'joint_type': 'fixed', 'joint_axis': [0.0, 0.0, -1.0], 'origin': [0.008, 0.0225, 0.0625], 'rpy': [0.0, -0.0, 0.0], 'child': 'sensor_imu'}, {'joint_name': 'wheel_left', 'joint_type': 'continuous', 'joint_axis': [1.0, -0.0, -0.0], 'origin': [-0.031, 3.85118e-32, 0.0225], 'rpy': [-2.5241e-48, -1.2326e-32, 5.25487e-46], 'child': 'wheel_1'}, {'joint_name': 'wheel_right', 'joint_type': 'continuous',

3526

In [5]:
import mujoco as mj

spec = mj.MjSpec.from_file(OUTPUT)
model = spec.compile()
data = mj.MjData(model)

mj.mj_forward(model, data)
r = model.body("chassis")
print(r.ipos, r.inertia)
r_ = spec.body("chassis")
r_.ipos = [0, 0, 0.017]
r_.mass = r.mass[0]
r_.inertia = r.inertia
r_.explicitinertial = True
print(r_.ipos, r_.mass, r_.inertia)
spec.encode("test.xml")


[-6.47678578e-19  4.74964290e-18  4.59784047e-02] [5.87216000e-05 5.14595315e-05 4.92017057e-05]
[0.    0.    0.017] 0.127 [5.87216000e-05 5.14595315e-05 4.92017057e-05]


3526

In [6]:
import mujoco as mj

for name in ["robot", "robot-bala2-mj", "bala2ish", "bala2ish-com"]:
	spec = mj.MjSpec.from_file(f"{name}.xml")
	model = spec.compile()
	data = mj.MjData(model)

	mj.mj_forward(model, data)

	robot = data.body("root")

	com = robot.subtree_com * 1000

	axlepos = (data.body("wheel").xpos + data.body("wheel_1").xpos) / 2

	print(name)
	print(f"  mass {model.body('root').subtreemass[0] * 1000:.4f} gram")
	print(f"  center of mass [ {com[0]:.4f} {com[1]:.4f} {com[2]:.4f} ] mm")
	print(
		f"  axle position {axlepos[2] * 1000:.4f} mm [ {axlepos[0]:.4f} {axlepos[1]:.4f} {axlepos[2]:.4f} ]"
	)
	print(
		f"  pendulum arm {(robot.subtree_com[2] - data.body('wheel').xpos[2]) * 1000:.4f} mm"
	)

	if name == "robot":
		site = data.site("site_sensor_imu")
	elif name == "robot-bala2-mj":
		site = data.site("imu_site")
	relpos = (site.xpos - axlepos) * 1000
	print(
		f"  IMU pos relative to axle [ {relpos[0]:.4f} {relpos[1]:.4f} {relpos[2]:.4f} ] mm"
	)

	print()

robot
  mass 170.5314 gram
  center of mass [ 0.0000 0.0000 37.4685 ] mm
  axle position 22.5000 mm [ 0.0000 0.0000 0.0225 ]
  pendulum arm 14.9685 mm
  IMU pos relative to axle [ 13.0000 7.0000 48.4628 ] mm

robot-bala2-mj
  mass 161.0000 gram
  center of mass [ 0.0000 0.0000 35.9099 ] mm
  axle position 22.5000 mm [ 0.0000 0.0000 0.0225 ]
  pendulum arm 13.4099 mm
  IMU pos relative to axle [ -8.0000 22.5000 40.0000 ] mm

bala2ish
  mass 161.0000 gram
  center of mass [ -0.0000 0.0000 41.0202 ] mm
  axle position 22.5000 mm [ 0.0000 0.0000 0.0225 ]
  pendulum arm 18.5202 mm
  IMU pos relative to axle [ -8.0000 22.5000 40.0000 ] mm

bala2ish-com
  mass 161.0000 gram
  center of mass [ -0.0000 0.0000 35.9099 ] mm
  axle position 22.5000 mm [ 0.0000 0.0000 0.0225 ]
  pendulum arm 13.4099 mm
  IMU pos relative to axle [ -8.0000 22.5000 40.0000 ] mm



# Launch MuJoCo

In [ ]:
import mujoco as mj
import mujoco.viewer

spec = mj.MjSpec.from_file("bala2ish.xml")
model = spec.compile()
data = mj.MjData(model)

mujoco.viewer.launch(model, data)

/nix/store/8w97xwciwfg8zs6yp7a1dfyflz9y2slh-python3-3.13.15-env/lib/python3.13/site-packages/glfw/__init__.py:917: GLFWError: (65548) b'Wayland: The platform does not provide the window position'
  warnings.warn(message, GLFWError)


In [9]:
import time

import mujoco as mj
import mujoco.viewer as mjv

from types import SimpleNamespace
from scipy.spatial.transform import Rotation
import numpy as np

spec = mj.MjSpec.from_file("bala2ish-0.15N.xml")
model = spec.compile()
data = mj.MjData(model)

try:
	viewer.close()
except:
	pass

step = 0

a1 = data.actuator("act_wheel_left")
a2 = data.actuator("act_wheel_right")
jv1 = data.sensor("jointvel_wheel_left")
jv2 = data.sensor("jointvel_wheel_right")
robot = data.body("root")

d = SimpleNamespace(
			robot=data.body("root"),  # to get position
			act1=data.actuator("act_wheel_left"),
			act2=data.actuator("act_wheel_right"),
			imu_gyro=data.sensor("imu_gyro"),
			imu_quat=data.sensor("imu_quat"),
			priv_vel=data.sensor("priv_vel"),
			input="voltage"
		)


a1.ctrl = -1
# a2.ctrl = 1

mj.mj_forward(model, data)

vels = []

viewer = mjv.launch_passive(model, data)

while True:
	step += 1
	step_start = time.time()
	mj.mj_step(model, data)
	viewer.sync()

	gyro = d.imu_gyro.data
	quat = d.imu_quat.data

	# r = Rotation.from_quat(quat, scalar_first=True)
	# r.inv()

	# eul = r.as_euler("xyz", degrees=False, suppress_warnings=True)
	if step % 30 == 0:
		print(step, "vel", jv1.data[0], jv2.data[0])
		# print(step, "eul", eul)
		# print(" " * (len(f"{step}")), "vel", d.priv_vel.data)


	# if step == 400:
	# 	pos = robot.xpos[1]
	# if step > 400:
	# 	vels.append(jv1.data[0])
	# if step > 800:
	# 	print("moved", (robot.xpos[1] - pos)/2, "m/s")
	# 	print("avg vel steps 400-800", np.mean(vels)/(2*np.pi), "rotations per second")
	# 	viewer.close()
	# 	time.sleep(1)
	# 	break

	# print(step, j1.qvel, j1.qacc)

	sleep = 0.005 - (time.time() - step_start)
	if sleep > 0:
		time.sleep(sleep)


30 vel -139.95115258547892 1.4138259692215047e-18
60 vel -148.50105948447862 -7.843888291320157e-18
90 vel -148.97248911698304 -3.6068470045659865e-17
120 vel -148.9984831121768 -2.1849059018407202e-17
150 vel -148.9999163334508 -1.3023972003833093e-17
180 vel -148.99999541530923 -7.546799706362751e-18
210 vel -148.99999972124 -3.48656302245186e-17
240 vel -149.0000000046917 -2.1102532358371374e-17
270 vel -148.99999998865707 -1.2560650421811486e-17
300 vel -149.00000000103762 -7.259245760691202e-18
330 vel -149.00000000854453 -3.370139177825998e-17
360 vel -148.99999998291466 -2.037996337644617e-17
390 vel -149.00000002344686 -1.2112197202585654e-17
420 vel -148.99999997317852 -6.98091902023208e-18
450 vel -149.00000002677902 -3.2574511803809814e-17
480 vel -148.99999997667499 -1.968057980060393e-17
510 vel -149.00000001689938 -1.167813411596343e-17
540 vel -148.99999999167918 -6.711523832971235e-18
570 vel -148.99999999868223 -3.148379720838193e-17
600 vel -149.0000000107885 -1.90036

KeyboardInterrupt: 

In [44]:
viewer.close()